### Create a config, pass it to the dataset class, then build it

In [1]:
from utils.swbd_disfluency_loader import SwbdDisfluencyDataset, Config
from collections import Counter

cfg = Config(
    zayats_tsv = "data/raw/zayats/switchboard_corrected_with_silver_reannotation.tsv",
    nxt_dir = "data/raw/nxt/nxt_switchboard_ann/xml", # ... or anything else you want to override
)
ds = SwbdDisfluencyDataset(cfg).build()

Processing convos . . .: 100%|██████████| 1284/1284 [03:03<00:00,  7.00it/s]


built 174328 records from 193085 rows
alignments: 1284 | median agreement 0.985 | <0.9 (skipped enrichment): 0 | A/B swaps: 90


### You can use it as is

In [2]:
print(f"--- Total: {len(ds)}")
print(ds[0])

--- Total: 174328
{'conv': 'sw2005', 'file': 'sw2005.trans', 'speaker': 'A', 'turn': 1, 'sent_num': 0, 'nxt_speaker': 'A', 'align_agreement': 0.984, 'swapped': False, 'enriched': True, 'tokens': ['okay'], 'bio': ['O'], 'typ': [None], 'sil_after': [], 'fluent': 'okay', 'disfluent': 'okay', 'fluent_tokens': ['okay'], 'disfluent_tokens': ['okay'], 'disfl_labels': [None], 'n_tokens_fluent': 1, 'n_tokens_disfluent': 1, 'n_disfl_tokens': 0}


### Splitting (adds new col of `split` as train/val/test)

In [3]:
ds.add_splits(ratios=(0.8, 0.1, 0.1))

add_splits (by conv): convs {'train': 901, 'val': 113, 'test': 112} | records {'train': 140456, 'val': 17299, 'test': 16573}


### Adding context (before filter_records so short utterances remain available as context.)

In [4]:
# Adding fluent cleaned context (k=3 preceding fluent non empty utterances) to each record
ds.add_context(k=3,
                source="fluent",
                include_other_speaker=True,
                speaker_prefix=True, join=" ", 
                template="{speaker}: {text}"
                )

add_context: k=3 source=fluent other_speaker=True | avg turns attached 2.96


### Post filtering if needed

In [5]:
ds.filter_records(min_tokens_fluent=10)

filter_records: kept 38525/174328


### Saving/Loading through .jsonl

In [6]:
# Save it
ds.save_jsonl("swbd_disfluency.jsonl")
del ds

In [7]:
# Load it
ds = SwbdDisfluencyDataset.from_jsonl("swbd_disfluency.jsonl")
n = len(ds)
rec_counts = Counter(r.get("split", "<none>") for r in ds.records)
conv_counts = {s: len({r["conv"] for r in ds.records if r.get("split") == s})
               for s in rec_counts}

print(f"--- Total records: {n}")
for s in ("train", "val", "test"):
    if s in rec_counts:
        c = rec_counts[s]
        print(f"  {s:5}: {c:6,} records ({c/n:6.1%})  |  {conv_counts[s]:4} convs")
        
print(ds[0])

--- Total records: 38525
  train: 30,906 records ( 80.2%)  |   901 convs
  val  :  3,689 records (  9.6%)  |   113 convs
  test :  3,930 records ( 10.2%)  |   112 convs
{'conv': 'sw2005', 'file': 'sw2005.trans', 'speaker': 'A', 'turn': 1, 'sent_num': 1, 'nxt_speaker': 'A', 'align_agreement': 0.984, 'swapped': False, 'enriched': True, 'tokens': ['uh', 'first', 'um', 'i', 'need', 'to', 'know', 'uh', 'h-', 'how', 'do', 'you', 'feel', 'about', 'uh', 'about', 'sending', 'um', 'an', 'elderly', 'uh', 'family', 'member', 'to', 'a', 'nursing', 'home'], 'bio': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'BE_IP', 'C', 'O', 'O', 'O', 'BE_IP', 'C', 'C', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], 'typ': ['filled_pause', None, 'filled_pause', None, None, None, None, 'filled_pause', None, None, None, None, None, None, 'filled_pause', None, None, 'filled_pause', None, None, 'filled_pause', None, None, None, None, None, None], 'sil_after': [0, 1, 2, 6, 7, 13, 14, 16, 17, 19, 20], 'fluent': 'fi

### Get it in another format

In [8]:
# Pandas DataFrame
df_full = ds.to_pandas(split='train', enriched_only=True)
df_full.head(5)

,conv,file,speaker,turn,sent_num,nxt_speaker,align_agreement,swapped,enriched,tokens,...,fluent_tokens,disfluent_tokens,disfl_labels,n_tokens_fluent,n_tokens_disfluent,n_disfl_tokens,split,context_turns,n_context_turns,context
0,sw2008,sw2008.trans,B,2,2,B,0.976,False,True,"[um, actually, i, wore, corduroy, shorts, with...",...,"[actually, i, wore, corduroy, shorts, with, a,...","[um, <sil>, actually, i, wore, corduroy, short...","[filled_pause, silent_pause, None, None, None,...",12,17,5,train,"[{'speaker': 'A', 'text': 'what did you wear t...",3,A: what did you wear to work today B: let's se...
1,sw2008,sw2008.trans,B,16,1,B,0.976,False,True,"[um, you, know, there, 's, real, no, no, no, r...",...,"[you, know, there, 's, no, dress, code, where,...","[um, <sil>, you, know, <sil>, there, 's, real,...","[filled_pause, silent_pause, None, None, silen...",10,18,8,train,"[{'speaker': 'B', 'text': 'uh-huh'}, {'speaker...",3,B: uh-huh B: uh-huh B: uh-huh
2,sw2008,sw2008.trans,B,16,3,B,0.976,False,True,"[i, um, do, n't, like, to, wear, heels, that, ...",...,"[i, do, n't, like, to, wear, heels, that, real...","[i, <sil>, um, <sil>, do, n't, like, to, wear,...","[None, silent_pause, filled_pause, silent_paus...",12,16,4,train,"[{'speaker': 'B', 'text': 'uh-huh'}, {'speaker...",3,B: uh-huh B: you know there's no dress code wh...
3,sw2008,sw2008.trans,B,18,2,B,0.976,False,True,"[in, the, summer, i, you, know, i, like, one, ...",...,"[in, the, summer, you, know, i, like, one, pie...","[in, the, summer, i, <sil>, you, know, <sil>, ...","[None, None, None, reparandum, silent_pause, N...",15,20,5,train,"[{'speaker': 'A', 'text': 'yeah'}, {'speaker':...",3,A: yeah B: so i predominantly wear flat shoes ...
4,sw2008,sw2008.trans,A,31,1,A,0.984,False,True,"[do, you, like, to, wear, jeans, most, of, the...",...,"[do, you, like, to, wear, jeans, most, of, the...","[do, you, like, to, wear, jeans, most, of, the...","[None, None, None, None, None, None, None, Non...",10,10,0,train,"[{'speaker': 'B', 'text': 'accessories'}, {'sp...",3,B: accessories B: uh-huh A: what about your ca...


In [9]:
# HuggingFace datasets.Dataset
# hf_full = ds.to_hf_dataset(fields=("fluent","disfluent"))
hf_full = ds.to_hf_dataset(fields=("context", "fluent", "disfluent"), split="train", enriched_only=True)

print(hf_full)
print(hf_full[0])

Dataset({
    features: ['context', 'fluent', 'disfluent'],
    num_rows: 17332
})
{'context': "A: what did you wear to work today B: let's see B: what did i wear to work today", 'fluent': 'actually i wore corduroy shorts with a white blouse and flat shoes', 'disfluent': 'um <sil> actually i wore corduroy shorts with a white blouse <sil> um <sil> and flat shoes'}


In [10]:
# FYI you dont have to use any of exports' flags to filter or
# anything like that, you can use pandas df and work with it as you like,
# all columns are there.